# Supernovae Lab 2

### Goals: 

In this second lab, we will:


* Become familiar with the `step_SN` parameters (kicks, SN perscriptions) in POSYDON and explore how they influence the evolution of a binary system

* Learn about the available `SN_MODELS` (i.e. set of parameter assumptions)  in POSYDON

* Extract stellar profiles and understand how a black hole spin is calculated


## 1. Supernova effects on one binary system


We will evolve one binary system using the default POSYDON `.ini` file. In this case, let’s consider a binary with the following initial parameters: $M_{1,init} = 28.15  M_{\odot}$, $M_{2,init} = 19.7  M_{\odot}$, and $P_{\text{orb,init}} = 10$ days.

Let's define the initial values here:


In [ ]:
m1_init = 28.15 # in Msun
m2_init = 19.7 # in Msun
p_init = 10 # in days

and let's load the steps from `.ini` file:

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

import os
import shutil
import pandas as pd

from posydon.config import PATH_TO_POSYDON
from posydon.popsyn.io import simprop_kwargs_from_ini
from posydon.binary_evol.simulationproperties import SimulationProperties
from posydon.binary_evol.singlestar import SingleStar
from posydon.binary_evol.binarystar import BinaryStar

from posydon.binary_evol.MESA.step_mesa import MS_MS_step
from posydon.binary_evol.MESA.step_mesa import CO_HeMS_step
from posydon.binary_evol.MESA.step_mesa import CO_HMS_RLO_step
from posydon.binary_evol.MESA.step_mesa import CO_HeMS_RLO_step
from posydon.binary_evol.DT.step_detached import detached_step
from posydon.binary_evol.DT.step_disrupted import DisruptedStep
from posydon.binary_evol.DT.step_merged import MergedStep
from posydon.binary_evol.DT.step_initially_single import InitiallySingleStep
from posydon.binary_evol.SN.step_SN import StepSN

# Get the environment variable PATH_TO_POSYDON
base_dir = os.environ["PATH_TO_POSYDON"]
path_to_ini = os.path.join(base_dir, "posydon/popsyn/population_params_default.ini")

sim_prop = SimulationProperties.from_ini(path_to_ini, load_steps=True, metallicity=1, verbose=True)

### Assigning a Specific Vector for CC1 Kick to Ensure Deterministic Evolution

By default, POSYDON has historically sampled the supernova kick magnitude from a Maxwellian distribution with a random direction (although the default was recently changed to the log-normal distribution from Disberg & Mandel 2025). To make the system’s evolution deterministic, we instead specify a fixed kick vector.

This is controlled through the following four elements/columns:


`natal_kick_velocity`: kick magnitude (km/s)

`natal_kick_azimuthal_angle`: azimuthal angle (rad)

`natal_kick_polar_angle`: polar angle (rad)

`natal_kick_mean_anomaly`: orbital mean anomaly (rad) (relevant only for eccentric orbits)

For further details on evolving specific binaries with a fixed kick, see the documentation:  

https://posydon.org/POSYDON/latest/tutorials-examples/population-synthesis/evolve_single_binaries.html

In [ ]:
dict_initial_binary = {'time': 0.0, 'state': 'detached', 'event': 'ZAMS', 'eccentricity': 0.0}

STAR1, STAR2 = SingleStar(**{'mass': m1_init, 'state': 'H-rich_Core_H_burning', 'natal_kick_velocity': 41}), \
               SingleStar(**{'mass': m2_init, 'state': 'H-rich_Core_H_burning'})
binary = BinaryStar(STAR1, STAR2, **{**dict_initial_binary, 'orbital_period': p_init}, properties=sim_prop)


### Evolving the binary system!

Next, evolve the binary system and specify the `scalar_names` for both the primary and secondary stars that you wish to include in the output. For (most of) the options available see: `population_params_default.ini`.
You can track its evolution with the `plot_SN_evolution` function provided for this lab. This function visualizes how the system evolves over time by plotting the total masses, helium core masses of the binary components as well as the orbital period, and eccentricity of the binary system.



In [ ]:
S1_kwargs = {
    "only_select_columns": ["spin"],
    "scalar_names": ["natal_kick_velocity", 
                     "natal_kick_azimuthal_angle",
                     "natal_kick_polar_angle", 
                     "natal_kick_mean_anomaly", 
                     "SN_type", 
                     "f_fb",
                     "h1_mass_ej", 
                     "he4_mass_ej",
                     "avg_c_in_c_core_at_He_depletion", 
                     "co_core_mass_at_He_depletion"]
}

S2_kwargs = S1_kwargs.copy()
binary.evolve()

df=binary.to_df(extra_columns={'step_names':'string'})
df_oneline=binary.to_oneline_df(S1_kwargs=S1_kwargs, S2_kwargs=S2_kwargs)

Display the oneline dataframe columns saved:

In [ ]:
df_oneline.columns

In this Lab, we will make use of `plot_SN_evolution` for visualization, that can be found locally, to visualize the time evolution of a system, focusing on its CC1/2 properties.

In [ ]:
from plotting_evolution_till_CC_one_system  import plot_SN_evolution # a function made for this lab to visualize the time evolution of a systems, focusing on its CC1/2 properties
plot_SN_evolution(df, df_oneline)
col=["step_names","state","time", "S1_state", "S2_state","event"]
df[col]

When visualizing the binary’s evolution over time, POSYDON outputs only final values (the outcome of a step) of each evolutionary step. Consequently, the detailed history of mass transfer and binary interactions is not resolved. Alongside this, we also show key properties of the system, including the nature of the compact object formed after `CC1` or `CC2`, whether the system remains bound or becomes disrupted following compact object formation, as well as the natal kick imparted to the newly formed BH or NS and, in the case of a BH, its spin.

<div class="alert alert-success">

## Exercise: 
To become more familiar with commonly used supernova parameters and where they are stored (in the `history` or `oneline` keys), print key information for the system above. This should include the fallback fraction, the amounts of hydrogen and helium ejected during the SN event,  the kick magnitude at `CC1`, and whether the system was disrupted or remained bound after `CC1`.
   
</div>

In [ ]:
print(### fill in ###)
print(### fill in ###)
print(### fill in ###)
print(### fill in ###)
print(### fill in ###)

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint 1</summary></b>

All this information is stored as columns in `df_oneline` except the `step_names` stored in `df`. The corresponding columns are organized as follows:

`f_fb`: fallback fraction

`h1_mass_ej`: hydrogen mass in the ejecta

`he4_mass_ej`: helium mass in the ejecta

magnitude of the kick: `S1_natal_kick_velocity`

The state of the binary after `CC1` is recorded in the `step_names` column at `step_SN` (i.e., the outcome of `step_SN`). NOT at the `CC1` or `CC2` event, (which calls the `step_SN`)

    
</details>

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution</summary></b>
```python
print("fallback fraction:", df_oneline["S1_f_fb"].values)
print("H mass in the ejecta in solar masses",df_oneline["S1_h1_mass_ej"].values)
print("He mass in the ejecta in solar masses", df_oneline["S1_he4_mass_ej"].values)
print("kick magnitude:",df_oneline["S1_natal_kick_velocity"].values)
post_CC1 = (df["step_names"] == "step_SN") & (df["event"].shift(1) == "CC1")
print("state of the binary after CC1:", df.loc[post_CC1, "state"].values)
```
    
</details>

## 2. `step_SN` options in POSYDON

Now, let's explore the impact of modifying the parameters available for `step_SN` in the `.ini` file. First, we'll review the options available. For a more detailed explanation of each parameter, refer to the POSYDON documentation for `step_SN`:    
https://posydon.org/POSYDON/latest/_modules/posydon/binary_evol/SN/step_SN.html


```python

[step_SN]
  import = ['posydon.binary_evol.SN.step_SN', 'StepSN']
    # builtin posydon step
  absolute_import = None
    # 'package' kwarg for importlib.import_module
  
  # -------------------------------
  # Explosion Mechanisms
  # -------------------------------
  mechanism = 'Fryer+12-delayed'
    # v2 interpolators support: 'Fryer+12-rapid', 'Fryer+12-delayed',
    #                          'Sukhbold+16-engine', 'Patton&Sukhbold20-engine'
    # need profiles: 'direct'
  engine = ''
    # 'N20' or 'W20' for 'Sukhbold+16-engine', 'Patton&Sukhbold20-engine'
    # '' for the others
  ECSN = "Tauris+15"
    # "Tauris+15", "Podsiadlowski+04"

  # -------------------------------
  # Mass and neutrino loss
  # -------------------------------
  conserve_hydrogen_envelope = False
    # True, False
  max_neutrino_mass_loss = 0.5
    # float (0,inf)
    # v2 interpolators support: 0.5
  max_NS_mass = 2.5
    # float (0,inf)
    # v2 interpolators support: 2.5

  # -------------------------------
  # P(P)ISN options 
  # -------------------------------
  PISN = "Hendriks+23"
    # v2 interpolators support: "Hendriks+23"
    # other options: None, "Marchant+19"
  PISN_CO_shift = 0.0
    # Only when using Hendriks+23
    # float (-inf,inf)
    # v2 interpolators support: 0.0
  PPI_extra_mass_loss = -20.0
    # Only when using Hendriks+23
    # float (-inf,inf)
    # v2 interpolators support: 0.0 or -20.0
  conserve_hydrogen_PPI  = False
    # Only when using Hendriks+23
    # True, False


  # -------------------------------
  # interpolation options
  # -------------------------------
  use_interp_values = True
    # True, False
  use_profiles = True
    # True, False
  use_core_masses = True
    # True, False
  allow_spin_None = False
    # True, False
  approx_at_he_depletion = False
    # True, False


  # -------------------------------
  # kick options
  # -------------------------------
  kick = True
    # True, False
  kick_normalisation = 'one_over_mass'
    # "one_minus_fallback", "one_over_mass", "NS_one_minus_fallback_BH_one",
    # "one", "zero"
  sigma_kick_CCSN_NS = 265.0
    # float (0,inf)
  sigma_kick_CCSN_BH = 265.0
    # float (0,inf)
  sigma_kick_ECSN = 20.0
    # float (0,inf)
    
```    

### 2.1 Varying Supernova Kicks

Let’s draw kicks from a Maxwellian distribution, with dispersions of 60 km/s for NSs. BHs receive natal kicks with the same dispersion, scaled by the BH mass ($𝑀_{BH}$) with 1.4$M_{\odot}$ / $𝑀_{BH}$.

In [ ]:
# First, let’s revert the kick to be randomly drawn from its distribution:
STAR1, STAR2 = (SingleStar(**{'mass': m1_init, 'state': 'H-rich_Core_H_burning',
                             'natal_kick_velocity': None,
                             'natal_kick_azimuthal_angle': None,
                             'natal_kick_polar_angle': None,
                             'natal_kick_mean_anomaly': None}),
               SingleStar(**{'mass': m2_init, 'state': 'H-rich_Core_H_burning'}))
    
binary = BinaryStar(STAR1, STAR2, **{**dict_initial_binary, 'orbital_period': p_init}, properties=sim_prop)

In [ ]:
sim_prop.load_a_step("step_SN", (StepSN, {'sigma_kick_CCSN_NS': 60, 'sigma_kick_CCSN_BH': 60}))
binary = BinaryStar(STAR1, STAR2, **{**dict_initial_binary, 'orbital_period': p_init}, properties=sim_prop)
binary.evolve()

df_oneline = binary.to_oneline_df(S1_kwargs=S1_kwargs, S2_kwargs=S2_kwargs)
df = binary.to_df(extra_columns={'step_names':'string'})
plot_SN_evolution(df, df_oneline)

Let’s now draw kicks from a Maxwellian distribution, with dispersions of 20 km/s for NSs. BHs receive natal kicks with the same dispersion, scaled by the BH mass ($𝑀_{BH}$) with 1.4 $M_{\odot}$ / $𝑀_{BH}$.

In [ ]:
sim_prop.load_a_step("step_SN", (StepSN, {'sigma_kick_CCSN_NS': 20, 'sigma_kick_CCSN_BH': 20}))
binary = BinaryStar(STAR1, STAR2, **{**dict_initial_binary, 'orbital_period': p_init}, properties=sim_prop)
binary.evolve()
df_oneline=binary.to_oneline_df(S1_kwargs=S1_kwargs, S2_kwargs=S2_kwargs)
df=binary.to_df(extra_columns={'step_names':'string'})
plot_SN_evolution(df, df_oneline)

<div class="alert alert-success">

## Exercise:
Examine how lower supernova kicks affect the orbital parameters, specifically eccentricity and orbital period, after `CC1`. Are there any differences in the orbital parameters before `CC1` for different SN kick values? Report the orbital separation immediately after `CC1` and determine whether the system remains bound.

</div>







In [ ]:
# Repeat the previous steps for each binary you evolve. For each case, record the relevant values and, at the end, compare the results across different SN kick magnitudes.

print("Post-CC1 orbital period", ??)
print("Post-CC1 orbital eccentricity", ??)
print("Post-CC1 orbital separation", ??)

print("Pre-CC1 orbital period", ??)
print("Pre-CC1 orbital eccentricity",??)
print("Pre-CC1 orbital separation", ??)


print("Kick :", ??)

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint 1</summary></b>

– Report the orbital period just before step_SN (CC1) and check whether it changes when varying the kicks.

– Report the orbital period output from step_SN and examine how the orbital period, separation ["separation"], and eccentricity vary with different kicks. Also, report the state of the binary after step_SN to determine whether it becomes disrupted or remains bound.

– Repeat the evolution of each system with different kicks and keep all the appropriate information and then compare.


    
</details>

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution</summary></b>

 

```python
print("Post-CC1 orbital period",df.loc[(df["step_names"] == "step_SN") & (df["event"].shift(+1) == "CC1"), "orbital_period"].values[0])
print("Post-CC1 orbital eccentricity",df.loc[(df["step_names"] == "step_SN") & (df["event"].shift(+1) == "CC1"), "eccentricity"].values[0])
print("Post-CC1 orbital separation",df.loc[(df["step_names"] == "step_SN") & (df["event"].shift(+1) == "CC1"), "separation"].values[0])

print("Pre-CC1 orbital period",df.loc[(df["step_names"].shift(-1) == "step_SN") & (df["event"] == "CC1"), "orbital_period"].values[0])
print("Pre-CC1 orbital eccentricity",df.loc[(df["step_names"].shift(-1) == "step_SN") & (df["event"] == "CC1"), "eccentricity"].values[0])
print("Pre-CC1 orbital separation",df.loc[(df["step_names"].shift(-1) == "step_SN") & (df["event"] == "CC1"), "separation"].values[0])


print("Kick :",df_oneline["S1_natal_kick_velocity"].values[0])

# We do not report differences of the orbital parameters of the system just before the first step_SN as the kicks affect only the secondaries.

```

    
</details>

### Counting Disruptions After `CC1`

<div class="alert alert-success">

## Exercise:

1. Rerun a binary system 10 times. This time, we have adjusted the initial parameters so that the primary evolves into a neutron star. Choose any value for the velocity dispersion of the Maxwellian natal-kick distribution for a neutron star (`sigma_kick_NS`) using the script provided below (simply set the value of `sigma_kick_NS`). In each trial, the actual kick should still be randomly drawn from the distribution. How many of the 10 runs lead to system disruption?

2. Repeat the experiment with several different choices of `sigma_kick_NS`. Can you identify an approximate value of `sigma_kick_NS` that results in disruption in about half of the trials?

3. Compute the Keplerian orbital velocity of the binary before first core collapse (`CC1`).

4. For the disrupted systems, compare the natal kick velocities with the pre-CC orbital velocity. What trends or patterns do you observe?


</div>







In [ ]:
import numpy as np
from posydon.utils import constants as cs #cgs

m1_init = 13 # in Msun 
m2_init = 10 # in Msun
p_init = 100 # in days
STAR1, STAR2 = SingleStar(**{'mass': m1_init, 'state': 'H-rich_Core_H_burning'}), \
               SingleStar(**{'mass': m2_init, 'state': 'H-rich_Core_H_burning'})


sigma_kick_NS =  60 # please set this value!
n_repeat = 20
n_o_disruptions = 0
kicks_at_disruption = []

sim_prop.load_a_step("step_SN", (StepSN, {'sigma_kick_CCSN_NS': sigma_kick_NS})) 
for i in range(n_repeat):
    binary = BinaryStar(STAR1, STAR2, **{**dict_initial_binary, 'orbital_period': p_init}, properties=sim_prop)
    binary.evolve()
    df_oneline=binary.to_oneline_df(S1_kwargs=S1_kwargs, S2_kwargs=S2_kwargs)
    df=binary.to_df(extra_columns={'step_names':'string'})
    
    mask_CC1 = (df['step_names'] == "step_SN") & (df['event'].shift(1) == "CC1") 
    v_kick_CC1 = df_oneline["S1_natal_kick_velocity"].values[0] #?
    state_after_CC1 = df.loc[mask_CC1, "state"].values[0] #?
    
    print(f"trial {i}: v_kick_CC1={v_kick_CC1:.0f} km/s, state_after_CC1={state_after_CC1}")
    
    if state_after_CC1 == "disrupted":
        n_o_disruptions += 1
        kicks_at_disruption.append(v_kick_CC1)
        
print("disruption percentage", n_o_disruptions/n_repeat*100., r"\%")

if kicks_at_disruption:
    median_kick = np.median(kicks_at_disruption)
    print(f"median kick at disruption = {median_kick:.0f} km/s")
    
else:
    print("no disruptions, average kick undefined")

In [ ]:
# Step 3

import numpy as np
from posydon.utils import constants as cs #cgs

G = cs.standard_cgrav

??

v =  ??

print("v_orbital_velocity_preCC1 (km/s):", ?? )

<div class="alert alert-warning" style="margin-top: 20px">

<details>
<summary><b>Hint for step 3</b></summary>
For a circular binary with masses \(m_1, m_2\), total mass \(M=m_1+m_2\), and separation \(a\), the relative orbital velocity is


$$
v = \sqrt{\frac{GM}{a}}
$$

</details>


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution</summary></b>

```python

import numpy as np
from posydon.utils import constants as cs #cgs

G = cs.standard_cgrav

mask_CC1 = df['event'] == "CC1" #& df['step_names'].shift(-1) == "CC1"
m1 = df.loc[mask_CC1, "S1_mass"] * cs.Msun
m2 = df.loc[mask_CC1, "S2_mass"] * cs.Msun
a  = df.loc[mask_CC1, "separation"] * cs.Rsun

Mtot   = m1 + m2
v =  np.sqrt((G*Mtot) / a)  # cm/s

print("v_orbital_velocity_preCC1 (km/s):", (v/1e5).values[0])
```

</details>

### 2.2 Different SN prescriptions

We evolve the binary below, with unrealistc high kicks, to force (almost deterministically) a disruption after CC1

In [ ]:
m1_init = 21 # in Msun 
m2_init = 17 # in Msun
p_init = 100 # in days
STAR1, STAR2 = SingleStar(**{'mass': m1_init, 'state': 'H-rich_Core_H_burning'}), \
               SingleStar(**{'mass': m2_init, 'state': 'H-rich_Core_H_burning'})

unr_kicks = {'sigma_kick_CCSN_NS': 10000, 'sigma_kick_CCSN_BH': 10000} # unrealistically high kicks
#sim_prop.load_a_step("step_SN", (StepSN, {**unr_kicks, 'mechanism': 'Fryer+12-delayed', 'engine':''})) 
sim_prop.load_a_step("step_SN", (StepSN, {**unr_kicks})) 

binary = BinaryStar(STAR1, STAR2, **{**dict_initial_binary, 'orbital_period': p_init}, properties=sim_prop)

binary.evolve()

df_oneline=binary.to_oneline_df(S1_kwargs=S1_kwargs, S2_kwargs=S2_kwargs)
df=binary.to_df(extra_columns={'step_names':'string'})

plot_SN_evolution(df, df_oneline)

<div class="alert alert-success">

## Exercise: Change the SN prescription
Evolve the same binary as above, but this time set `mechanism = Patton&Sukhbold20-engine` and `engine = N20` in `step_SN`. Report the differences you observe compared to the latest run.  What changed at CC1 / CC2? 
   
</div>

In [ ]:
### fill in ###

<div class="alert alert-warning" style="margin-top: 20px">

<details>
<summary><b>Hint</b></summary>

```python
sim_prop.load_a_step("step_SN",(StepSN, {'mechanism': 'XX', 'engine': 'XX'})) #fill in XX
```
</details>

<div class="alert alert-warning" style="margin-top: 20px">

<details>

<b><summary>Solution</summary></b>

```python
sim_prop.load_a_step("step_SN", (StepSN, {'mechanism': 'Patton&Sukhbold20-engine', 'engine':'N20'}))

binary = BinaryStar(STAR1, STAR2, **{**dict_initial_binary, 'orbital_period': p_init}, properties=sim_prop)

binary.evolve()

df_oneline=binary.to_oneline_df(S1_kwargs=S1_kwargs, S2_kwargs=S2_kwargs)
df=binary.to_df(extra_columns={'step_names':'string'})

plot_SN_evolution(df, df_oneline)

# Now the primary and secondary star form a neutron star instead of a black hole!

```
    
</details>

<div class="alert alert-success">

## Exercise:
In the previous example you found that the primary and secondary form a NS instead of a BH. Find out the parameters that determine the outcome of Patton & Sukhbold 2020 & Ertl 2016 prescription (e.g. core carbon abundances & carbon-oxygen core mass) and compare with the photo provided below.
   
</div>

In [ ]:
print("S1 avg C fraction at He depletion:", ??)
print("S1 CO core mass at He depletion:", ??)

print("S2 avg C fraction at He depletion:", ??)
print("S2 CO core mass at He depletion:", ??)

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution</summary></b>

```python 
print("S1 avg C fraction at He depletion:", df_oneline["S1_avg_c_in_c_core_at_He_depletion"].values[0])
print("S1 CO core mass at He depletion:", df_oneline["S1_co_core_mass_at_He_depletion"].values[0])

print("S2 avg C fraction at He depletion:", df_oneline["S2_avg_c_in_c_core_at_He_depletion"].values[0])
print("S2 CO core mass at He depletion:", df_oneline["S2_co_core_mass_at_He_depletion"].values[0])


```
    
</details>

Compare the values with the 2D parameter space for explodability estimate of Patton+2020  
<img src="./Patton_parameters.png" alt="Explodability estimate" width="40%">

<div class='alert alert-info'>
    
**Side Note:** In this lab, we have explored evolving a binary system using the `nearest_neighbour` interpolation method. However, POSYDON also allows running systems with other interpolation methods. One such method is the initial/final interpolation, referred to as `"linear3c_kNN"`. To use this method, set `interpolation_method = 'linear3c_kNN'` in all MESA steps.

Note that with this method, you cannot use `track_interpolation = True` or `use_profiles=True`, as those only work with `interpolation_method = 'nearest_neighbour'`. Instead, to determine the outcome of a supernova event with initial/final, you should set `use_interp_values=True`.  

In POSYDON there are available fixed SN_MODELS that can be used  for initial/final interpolation. 

# [Optional] 3. Profiles and rotating collapsing stars 

In the last part we want to briefly see the calculation of a black hole spin, based on the rotational profile of the collapsing star.  
First we start again with our default options for our run. However as we want to keep the profile, for now we need to change the interpolation method from the default to `interpolation_method ='nearest_neighbour'` so that we can have access to the stellar profile.  

In [ ]:
sim_prop.load_a_step("step_HMS_HMS", (MS_MS_step, {'track_interpolation':False, 'interpolation_method':'nearest_neighbour'}),metallicity=1.0)
sim_prop.load_a_step("step_CO_HeMS", (CO_HeMS_step, {'track_interpolation':False, 'interpolation_method':'nearest_neighbour'})metallicity=1.0)
sim_prop.load_a_step("step_CO_HMS_RLO", (CO_HMS_RLO_step, {'track_interpolation':False, 'interpolation_method':'nearest_neighbour'})metallicity=1.0)
sim_prop.load_a_step("step_CO_HeMS_RLO", (CO_HeMS_RLO_step, {'track_interpolation':False, 'interpolation_method':'nearest_neighbour'})metallicity=1.0)
sim_prop.load_a_step("step_SN", (StepSN, {'kick':False, 'use_interp_values':False})metallicity=1.0) # so that we redo the calculation ourselves

Let's run one close binary systems of very massive stars, with $M_{1} = 90  M_{\odot}$, $M_{2,init} = 35  M_{\odot}$, and $P_{\text{orb,init}} = 4$ days.

In [ ]:
m1_init = 90.00 # in Msun
m2_init = 35.00 # in Msun
p_init = 4. # in days

In [ ]:
STAR1, STAR2 = SingleStar(**{'mass': m1_init, 'state': 'H-rich_Core_H_burning'}), \
               SingleStar(**{'mass': m2_init, 'state': 'H-rich_Core_H_burning'})
binary = BinaryStar(STAR1, STAR2, **{**dict_initial_binary, 'orbital_period': p_init}, properties=sim_prop)
binary.evolve()

df_oneline = binary.to_oneline_df(S1_kwargs=S1_kwargs, S2_kwargs=S2_kwargs)
df = binary.to_df(extra_columns={'step_names':'string'})

In [ ]:
print(binary.star_1.profile)

Profile is found `None` for star_1 after evolution, as it is a BH at the end of the binary's evolution. But the profile of each star is kept at its preCC state.

The history of all properties in a single star object in POSYDON can be accessed as `binary.star_1/2.<propertyname>_history` 
So we can access the history of the profile of star_1 from `binary.star_1.profile_history`. 

In [ ]:
# Keep the profile in the evolution of star_1 at pre-CC1
mask_CC1 = np.array(df['event']== "CC1")
idx_CC1 = np.nonzero(mask_CC1)[0][0] 
profile = binary.star_1.profile_history[idx_CC1]

Let's see the columns of the profile

In [ ]:
profile.dtype.names

These follow the columns names and meanings as in MESA profiles. Documentation for kept profile columns : https://posydon.org/POSYDON/latest/components-overview/machine_learning/ProfileInterpolator.html

Now let's plot the rotational properties of the profile along the enclosed mass (and also along its radial coordinate)

In [ ]:
import matplotlib.pyplot as plt

# assuming profile is your structured numpy array
mass = profile["mass"]
radius = profile["radius"]
omega = profile["omega"]

# specific angular momentum j = omega * r^2
j = omega * radius**2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ---------- Subplot 1: vs Mass ----------
ax1 = axes[0]
l1, = ax1.plot(mass, omega, marker="o", linestyle="-", color="tab:blue", label=r"$\omega$")
ax1.set_xlabel("Enclosed Mass")
ax1.set_ylabel(r"$\omega$ (rad/s)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_title("Omega and Specific Angular Momentum vs Mass")

# twin y-axis for j
ax1b = ax1.twinx()
l2, = ax1b.plot(mass, j, marker="s", linestyle="--", color="tab:red", label=r"$j_{\rm eq}=\omega r^2$")
ax1b.set_ylabel(r"$j_{\rm eq}$ (cgs units)", color="tab:red")
ax1b.tick_params(axis="y", labelcolor="tab:red")

# combined legend
ax1.legend(handles=[l1, l2], loc="upper center")

# ---------- Subplot 2: vs Radius ----------
ax2 = axes[1]
l3, = ax2.plot(radius, omega, marker="o", linestyle="-", color="tab:blue", label=r"$\omega$")
ax2.set_xlabel("Radial coordinate")
ax2.set_ylabel(r"$\omega$ (rad/s)", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")
ax2.set_title("Omega and Specific Angular Momentum vs Radius")

# twin y-axis for j
ax2b = ax2.twinx()
l4, = ax2b.plot(radius, j, marker="s", linestyle="--", color="tab:red", label=r"$j=\omega r^2$")
ax2b.set_ylabel(r"$j_{\rm eq}$ (cgs units)", color="tab:red")
ax2b.tick_params(axis="y", labelcolor="tab:red")

plt.tight_layout()
plt.show()


Note that the outer layers maybe be having lower angular vvelocity but higher specific angular momentum due to being located far away from the rotational axis (assuming they are at the equator, else an azimuthal dependence is involved)

Here we will see how the dimensionless spin parameter, $\alpha$, is calculated in POSYDON. It is defined as the ratio between the black hole’s angular momentum $J$ and the maximum possible angular momentum for a black hole of mass $M$:

$
\alpha = \frac{c J}{G M^{2}}
$

where $c$ is the speed of light and $G$ is the gravitational constant. It takes values between $0 \leq \alpha \leq 1$, with $\alpha = 0$ corresponding to a non-rotating (Schwarzschild) black hole and $\alpha = 1$ to a maximally rotating (extremal Kerr) black hole. In principle, the spin can be used for a stellar object too (and it is kept as a value in the history of a singlestar object in POSYDON), with the possiblity of $>1$ values as there is not limit at the event horizon. 


In [ ]:
BH_spin = binary.star_1.spin_history[idx_CC1] #post-CC1
print(f"a_BH = {BH_spin:.3f}")

To calculate the BH spin at collaapse, we follow Batta & Ramirez 2019. We first assume the inner part to collapse fully, $2.51 M_{\odot}$ by default, with all of its angular momentum. After we collapse shell by shell, with part of the shell falling directly to the BH and part of it forrming an instataneous disk, with part of it eventually feeding mass and angular momentum to the BH.

<img src="./Batta_et_al_collapsing_shells.png" alt="Collapsing shells" width="30%">

We first restore the binary at pre-CC1 condition in order to be ready for the collapse

In [ ]:
binary.star_1.restore(idx_CC1) # https://posydon.org/POSYDON/latest/_modules/posydon/binary_evol/singlestar.html#SingleStar.restore
star_for_collapse =  binary.star_1

In [ ]:
from posydon.binary_evol.SN.profile_collapse import do_core_collapse_BH  #https://posydon.org/POSYDON/latest/api_reference/posydon.binary_evol.SN.html#posydon.binary_evol.SN.profile_collapse.do_core_collapse_BH
pre_CC_mass = star_for_collapse.mass
do_core_collapse_BH(star_for_collapse, mass_central_BH=2.51, mass_collapsing = pre_CC_mass,  neutrino_mass_loss = .5, verbose = True)

It is left as a homework exercise for whoever wants wants to go to the `do_core_collapse_BH` function within POSYDON (and the Batta+Ramirez paper) to understand the calculated parameters and the estimated of the energy that may be released as feedback from a jet creation.

<div class="alert alert-success">

## Exercise: 

Make the same calculation for the BH formed from star_2. Is there any disk formed during the slowly collapse of the rotating star?
   
</div>

In [ ]:
## Fill in ##

<div class="alert alert-warning" style="margin-top: 20px">

<details>
<summary><b>Hint</b></summary>
We need to restore star_2 at its state before its step_SN
</details>

<div class="alert alert-warning" style="margin-top: 20px">

<details>
<summary><b>Solution</b></summary>
    
```python
mask_CC2 = np.array(df['event']== "CC2")
idx_CC2 = np.flatnonzero(mask_CC2)[0]
profile = binary.star_2.profile_history[idx_CC2]
BH_spin = binary.star_2.spin_history[idx_CC2] #post-CC1
print(f"a_BH = {BH_spin:.3f}")
binary.star_2.restore(idx_CC2) 
star2_for_collapse =  binary.star_2
final_BH_mass = star2_for_collapse.mass
do_core_collapse_BH(star2_for_collapse, mass_central_BH=2.51, mass_collapsing = final_BH_mass,  neutrino_mass_loss = .5, verbose = True)
```
</details>